<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/09-user-defined-functions/01-pandas-udfs.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 9.1 — Pandas UDFs: vectorized escape hatches

Arrow batches instead of pickle, one Python call per batch instead
of per row. Requires pandas + pyarrow (`pip install "pyspark[sql]"`
brings both). We build all three shapes, step on the NaN ambush,
and put numbers on the speedup.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("9.1-pandas-udfs")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = (
    spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
    .withColumn("revenue", F.round(col("quantity") * col("unit_price"), 2))
)
orders.select("order_id", "customer_id", "product", "quantity", "revenue").show(5)

## Series → Series: the workhorse

Type hints declare the variant. Inside: pandas operations on the
whole batch — never a Python loop per element.

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("string")
def clean_code(s: pd.Series) -> pd.Series:
    return s.str.strip().str.upper().str.replace(" ", "_")
    # .str ops null-propagate like native functions — no None guard needed

@pandas_udf("double")
def margin(price: pd.Series, qty: pd.Series) -> pd.Series:
    return (price * qty * 0.27).round(2)

(orders
 .withColumn("code", clean_code("product"))
 .withColumn("margin", margin("unit_price", "quantity"))
 .select("product", "code", "margin")
 .show(5))

In [ ]:
# the plan node changed: ArrowEvalPython instead of BatchEvalPython.
# Still a codegen fence, still an optimizer black box.
orders.withColumn("code", clean_code("product")).explain()

## The race: 9.0's pickled UDF vs the pandas UDF

Same arithmetic three ways. The pandas UDF refunds pickle and
per-row calls; the native version also keeps Catalyst's eyes open.

In [ ]:
import time

def bench(label, df):
    start = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label:>16}: {time.perf_counter() - start:5.2f} s")

plain_udf = F.udf(lambda x: (x % 97) * 1.5 + 1.0, "double")

@pandas_udf("double")
def vec_udf(s: pd.Series) -> pd.Series:
    return (s % 97) * 1.5 + 1.0

big = spark.range(2_000_000)
bench("pickled udf", big.select(plain_udf("id")))
bench("pandas udf",  big.select(vec_udf("id")))
bench("native",      big.select((col("id") % 97) * 1.5 + 1.0))

## Iterator of Series: pay setup once

Anything expensive to build — a model, a lookup table, compiled
regexes — goes ABOVE the loop: once per task, not once per batch.

In [ ]:
from typing import Iterator

@pandas_udf("double")
def taxed(batches: Iterator[pd.Series]) -> Iterator[pd.Series]:
    vat = {"loaded": 1.19}          # pretend: load_model(), open connection...
    for batch in batches:           # ...then stream the batches through it
        yield (batch * vat["loaded"]).round(2)

orders.select("revenue", taxed("revenue").alias("gross")).show(5)

## Series → scalar: grouped aggregate

Here (and only here) the Series IS the whole group — group-wide
statistics are the point. Works in groupBy().agg and over
unbounded windows.

In [ ]:
@pandas_udf("double")
def iqr(v: pd.Series) -> float:
    return float(v.quantile(0.75) - v.quantile(0.25))

orders.groupBy("country").agg(
    F.count("*").alias("n"), iqr("revenue").alias("rev_iqr")).show()

In [ ]:
# same custom statistic as a window annotation — 8.0's trick, custom math
w_country = Window.partitionBy("country")
(orders
 .withColumn("country_iqr", iqr("revenue").over(w_country))
 .select("order_id", "country", "revenue", "country_iqr")
 .show(6))

## The NaN ambush

Classic pandas has no null for ints, so an int column WITH nulls
arrives as float64 + NaN. Watch quantity shape-shift while order_id
stays an int — and test nulls with .isna(), never `is None`.

In [ ]:
@pandas_udf("string")
def dtype_of(s: pd.Series) -> pd.Series:
    return pd.Series([str(s.dtype)] * len(s))

orders.select(
    dtype_of("order_id").alias("order_id_dtype"),      # no nulls -> int
    dtype_of("quantity").alias("quantity_dtype"),      # 2 nulls  -> float64!
).distinct().show()

In [ ]:
@pandas_udf("string")
def qty_band(q: pd.Series) -> pd.Series:
    out = pd.Series("single", index=q.index)
    out[q >= 5] = "multi"
    out[q >= 10] = "bulk"
    out[q.isna()] = None            # .isna(), not == None — NaN ambush
    return out

orders.select("quantity", qty_band("quantity")).distinct().show()

## Exercises

1. `s.map(lambda x: ...)` inside a scalar pandas UDF: rerun the race
   with that version and explain the result.
2. Write a grouped-agg pandas UDF for the revenue *median* per
   customer, then produce the same numbers with the native
   `F.median` (4.0+) / `F.percentile_approx`. Compare plans.
3. Predict, then verify with `dtype_of`: what dtype does
   `unit_price` (double, no nulls) arrive as? What about `country`
   (string, 3 nulls)?
4. Break the length contract: return `s.dropna()` from a scalar
   pandas UDF on quantity and read the error you get. Why can't
   Spark accept a shorter Series here, while 9.2's applyInPandas
   happily changes row counts?
5. Rebuild 9.0's exercise-1 log parser as a pandas UDF using
   `.str.extract`, and re-race all three contenders.

In [ ]:
# your exercise code here